In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

train = train.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x))
test = test.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x))

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (439140, 16)
Test shape: (188165, 15)


In [4]:
X = train.drop(columns=['id', 'PitNextLap'])
y = train['PitNextLap'].values
X_test = test.drop(columns=['id'])

categorical_features = ['Driver', 'Compound', 'Race']
continuous_features = [col for col in X.columns if col not in categorical_features]

cat_dims = []
for col in categorical_features:
    X[col] = X[col].fillna('Missing').astype(str)
    X_test[col] = X_test[col].fillna('Missing').astype(str)
    
    le = LabelEncoder()
    combined_data = pd.concat([X[col], X_test[col]], axis=0)
    le.fit(combined_data)
    
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])
    
    num_unique = len(le.classes_)
    cat_dims.append(num_unique)

scaler = StandardScaler()

for col in continuous_features:
    median_val = X[col].median()
    X[col] = X[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

X[continuous_features] = scaler.fit_transform(X[continuous_features])
X_test[continuous_features] = scaler.transform(X_test[continuous_features])

X_cat = X[categorical_features].values
X_cont = X[continuous_features].values
X_test_cat = X_test[categorical_features].values
X_test_cont = X_test[continuous_features].values

print(f"Cat dims: {cat_dims}")
print(f"Continuous features: {len(continuous_features)}")

Cat dims: [887, 5, 26]
Continuous features: 11


In [5]:
class F1TabularDataset(Dataset):
    def __init__(self, cat_data, cont_data, labels=None):
        self.cat_data = torch.tensor(cat_data, dtype=torch.long)
        self.cont_data = torch.tensor(cont_data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32) if labels is not None else None

    def __len__(self):
        return len(self.cont_data)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.cat_data[idx], self.cont_data[idx], self.labels[idx]
        return self.cat_data[idx], self.cont_data[idx]

class F1TabularNN(nn.Module):
    def __init__(self, cat_dims, num_conts):
        super(F1TabularNN, self).__init__()
        
        self.emb_layers = nn.ModuleList([
            nn.Embedding(num_cats, min(50, (num_cats + 1) // 2)) for num_cats in cat_dims
        ])
        
        total_emb_size = sum(min(50, (num_cats + 1) // 2) for num_cats in cat_dims)
        
        input_size = total_emb_size + num_conts
        
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(64, 1)
        )

    def forward(self, x_cat, x_cont):
        embeddings = [emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.emb_layers)]
        x_cat_emb = torch.cat(embeddings, dim=1)
        
        x = torch.cat([x_cat_emb, x_cont], dim=1)
        
        return self.network(x).squeeze()

In [6]:
BATCH_SIZE = 512
EPOCHS = 30
LR = 1e-3
PATIENCE = 5

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []

test_dataset = F1TabularDataset(X_test_cat, X_test_cont)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("coconut")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"Fold {fold+1}/{n_splits}")
    
    X_tr_cat, X_tr_cont, y_tr = X_cat[train_idx], X_cont[train_idx], y[train_idx]
    X_va_cat, X_va_cont, y_va = X_cat[val_idx], X_cont[val_idx], y[val_idx]
    
    train_loader = DataLoader(F1TabularDataset(X_tr_cat, X_tr_cont, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(F1TabularDataset(X_va_cat, X_va_cont, y_va), batch_size=BATCH_SIZE, shuffle=False)
    
    model = F1TabularNN(cat_dims, len(continuous_features)).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    
    best_val_auc = 0
    patience_counter = 0
    best_model_weights = None
    
    for epoch in range(EPOCHS):
        model.train()
        for b_cat, b_cont, b_labels in train_loader:
            b_cat, b_cont, b_labels = b_cat.to(device), b_cont.to(device), b_labels.to(device)
            
            optimizer.zero_grad()
            logits = model(b_cat, b_cont)
            loss = criterion(logits, b_labels)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_preds_fold = []
        with torch.no_grad():
            for b_cat, b_cont, _ in val_loader:
                b_cat, b_cont = b_cat.to(device), b_cont.to(device)
                logits = model(b_cat, b_cont)
                probs = torch.sigmoid(logits) # Convert logits to probabilities
                val_preds_fold.extend(probs.cpu().numpy())
                
        val_auc = roc_auc_score(y_va, val_preds_fold)

        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
            best_model_weights = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}. Best AUC: {best_val_auc:.5f}")
            break
            
    fold_scores.append(best_val_auc)
    print(f"Fold {fold+1} Best ROC AUC: {best_val_auc:.5f}\n")
    
    model.load_state_dict(best_model_weights)
    model.eval()
    
    val_preds_fold = []
    with torch.no_grad():
        for b_cat, b_cont, _ in val_loader:
            b_cat, b_cont = b_cat.to(device), b_cont.to(device)
            probs = torch.sigmoid(model(b_cat, b_cont))
            val_preds_fold.extend(probs.cpu().numpy())
    oof_preds[val_idx] = val_preds_fold
    
    test_preds_fold = []
    with torch.no_grad():
        for b_cat, b_cont in test_loader:
            b_cat, b_cont = b_cat.to(device), b_cont.to(device)
            probs = torch.sigmoid(model(b_cat, b_cont))
            test_preds_fold.extend(probs.cpu().numpy())
            
    test_preds += np.array(test_preds_fold) / n_splits

overall_auc = roc_auc_score(y, oof_preds)
print("-" * 30)
print(f"Mean Fold ROC AUC: {np.mean(fold_scores):.5f}")
print(f"Overall OOF ROC AUC: {overall_auc:.5f}")

coconut
Fold 1/5
Early stopping at epoch 22. Best AUC: 0.94023
Fold 1 Best ROC AUC: 0.94023

Fold 2/5
Early stopping at epoch 28. Best AUC: 0.93735
Fold 2 Best ROC AUC: 0.93735

Fold 3/5
Early stopping at epoch 23. Best AUC: 0.93846
Fold 3 Best ROC AUC: 0.93846

Fold 4/5
Early stopping at epoch 26. Best AUC: 0.93870
Fold 4 Best ROC AUC: 0.93870

Fold 5/5
Fold 5 Best ROC AUC: 0.93892

------------------------------
Mean Fold ROC AUC: 0.93873
Overall OOF ROC AUC: 0.93859


In [7]:
submission = pd.DataFrame({
    'id': test['id'],
    'PitNextLap': test_preds
})

submission.to_csv('submission.csv', index=False)